In [1]:
#pip install pandas

In [2]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [3]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [4]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [5]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [6]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [7]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [8]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [9]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 151,
 'tn': 2609,
 'fp': 28,
 'fn': 212,
 'misclassification_rate': 0.08,
 'false_positive_rate': 0.01061812665908229,
 'false_negative_rate': 0.5840220385674931}

### Check results on the test set (new data not yet seen by the model)

In [10]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 33,
 'tn': 854,
 'fp': 20,
 'fn': 93,
 'misclassification_rate': 0.113,
 'false_positive_rate': 0.02288329519450801,
 'false_negative_rate': 0.7380952380952381}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

According to VSC, (which I place little to know faith in given my current experience with this program, but according) the current model has a .113 misclassification rate, with a .738 false negative rate and a .023 false positive rate. Given there is a 10 percent chance of a misclassification, I hold very little confidence in this model's ability to predict a bot. Assuming this was used to bot check articles, or papers (a consistent concern of these predictors) I would fear very much that many articles and papers would slip through the cracks.

### What are potential ramifications of false positives from the model?

A false positive, which has a very low rate with my model, could needlessly weaken the credibility of non-bot sources. In an acedemic sense, it could result in a student recieving consequences for a crime they didn't commit.

### What are potential ramifications of false negatives from the model?

False negatives can lead to ai generated data or information being released into the world, which has several moral issues including the fact that the training data was taken without consent in almost all cases, but almost more importantly is often the information provided by ai is factually dubious. Missing an ai in this case can easily lead to false information leaking without a human presence to question. In school, this can lead to unearned high grades, which diminishes the grades of those who earned high grades.